# Cotton V1 Core Demo

End-to-end walkthrough of the benchmark + mill capacity + buy signal workflow.

In [ ]:
from pathlib import Path

import pandas as pd

from src.cotton_prices import PriceLoadConfig, load_cotton_prices
from src.benchmarks import BenchmarksConfig, compute_price_benchmarks, evaluate_spot_snapshot
from src.config_loader import load_mill_profiles, load_signal_config
from src.buy_rules import generate_signal_for_date

project_root = Path("..")
macrotrends_csv = project_root / "data" / "cotton_macrotrends_daily.csv"
worldbank_xlsx = project_root / "data" / "WBCOMM_Prices.xlsx"  # adjust if needed
mill_profiles_path = project_root / "config" / "mill_profiles.yml"
signals_path = project_root / "config" / "signals.yml"

In [ ]:
if not macrotrends_csv.exists():
    raise FileNotFoundError(
        f"MacroTrends file not found at {macrotrends_csv}. Place your daily export here or update the path."
    )

wb_path = str(worldbank_xlsx) if worldbank_xlsx.exists() else None

price_cfg = PriceLoadConfig(
    macrotrends_csv_path=str(macrotrends_csv),
    worldbank_xlsx_path=wb_path,
    fred_codes={"CPI": "CPIAUCSL"},
)
prices = load_cotton_prices(price_cfg)
prices.tail()

In [ ]:
bm_cfg = BenchmarksConfig()
prices_bm = compute_price_benchmarks(prices, config=bm_cfg)
snapshot = evaluate_spot_snapshot(prices_bm)
snapshot

In [ ]:
profiles = load_mill_profiles(mill_profiles_path)
mill = profiles["BD_Mill_25kSpindles_30Ne"]

signal_cfg = load_signal_config(signals_path)

decision = generate_signal_for_date(prices_bm, profile=mill, config=signal_cfg)
decision

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
prices_bm["cotton_spot_usd_per_lb"].plot(ax=ax, label="Spot")
ax.set_title("Cotton spot price with latest buy decision")
ax.legend();